In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visualitzar dades
pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")


df = pd.read_csv('../data/diabetic_data.csv')
print(df.shape)
df.head()

# Check
print(df.shape)
print("\n--- Target distribution ---")
print(df['readmitted'].value_counts())
print(df['readmitted'].value_counts(normalize=True).round(3))

In [ ]:
# Cambiem '?' per NaN
df = df.replace('?', np.nan)

# Check missingns per columna
missing = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
missing_summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct': missing_pct
}).query('missing_count > 0').sort_values('missing_pct', ascending=False)

# Check
print(missing_summary)

In [ ]:
# 1. Eliminem 'weight' (97% missing)
df = df.drop(columns=['weight'])

# 2. Eliminem pacients morts o a hospici (no poden ser readmessos en <30 dies)
death_hospice_codes = [11, 13, 14, 19, 20, 21]
df = df[~df['discharge_disposition_id'].isin(death_hospice_codes)]

# 3. Tractem >30 i no readmessos com iguals: (1 if readmitted within 30 days, 0 otherwise)
df['readmitted_binary'] = (df['readmitted'] == '<30').astype(int)

# 4. Convertim edat en valors numèrics
age_map = {
    '[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35,
    '[40-50)': 45, '[50-60)': 55, '[60-70)': 65, '[70-80)': 75,
    '[80-90)': 85, '[90-100)': 95
}
df['age'] = df['age'].map(age_map)

# 5. Convertim NaN informatius en la seva propia categoria
df['race'] = df['race'].fillna('Unknown')
df['payer_code'] = df['payer_code'].fillna('Unknown')
df['medical_specialty'] = df['medical_specialty'].fillna('Unknown')
df['max_glu_serum'] = df['max_glu_serum'].fillna('None')
df['A1Cresult'] = df['A1Cresult'].fillna('None')

In [ ]:
# Eliminem categories per al Hot Encoding
def group_icd9(code):
    """Group an ICD-9 code into one of 9 clinical categories."""
    if pd.isna(code):
        return 'Other'
    code = str(code)
    
    # V and E codes go to 'Other'
    if code.startswith('V') or code.startswith('E'):
        return 'Other'
    
    try:
        c = float(code)
    except ValueError:
        return 'Other'
    
    # Diabetes: 250.xx
    if 250 <= c < 251:
        return 'Diabetes'
    # Circulatory: 390-459, 785
    if (390 <= c < 460) or (int(c) == 785):
        return 'Circulatory'
    # Respiratory: 460-519, 786
    if (460 <= c < 520) or (int(c) == 786):
        return 'Respiratory'
    # Digestive: 520-579, 787
    if (520 <= c < 580) or (int(c) == 787):
        return 'Digestive'
    # Injury: 800-999
    if 800 <= c < 1000:
        return 'Injury'
    # Musculoskeletal: 710-739
    if 710 <= c < 740:
        return 'Musculoskeletal'
    # Genitourinary: 580-629, 788
    if (580 <= c < 630) or (int(c) == 788):
        return 'Genitourinary'
    # Neoplasms: 140-239
    if 140 <= c < 240:
        return 'Neoplasms'
    
    return 'Other'

for col in ['diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].apply(group_icd9)

# Check
print(df['diag_1'].value_counts())
print("\nAny missings left?", df[['diag_1', 'diag_2', 'diag_3']].isna().sum().sum())

In [ ]:
from sklearn.model_selection import train_test_split

# Eliminem les variables no necessaries restants
df = df.drop(columns=['encounter_id', 'readmitted'])

# Eliminem duplicats del mateix pacient
unique_patients = df.drop_duplicates('patient_nbr')['patient_nbr']
print(f"Total encounters: {len(df)}")
print(f"Unique patients: {len(unique_patients)}")
print(f"Avg encounters per patient: {len(df) / len(unique_patients):.2f}")

# Separem els pacients (80% train, 20% test)
train_patients, test_patients = train_test_split(
    unique_patients,
    test_size=0.2,
    random_state=42

train_df = df[df['patient_nbr'].isin(train_patients)].copy()
test_df = df[df['patient_nbr'].isin(test_patients)].copy()

# Eliminem variables que ja no necessitem
train_df = train_df.drop(columns=['patient_nbr'])
test_df = test_df.drop(columns=['patient_nbr'])

# Check
print(f"\nTrain shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Train positive rate: {train_df['readmitted_binary'].mean():.3f}")
print(f"Test positive rate: {test_df['readmitted_binary'].mean():.3f}")


In [ ]:
# Separate el resultat (y) de les altres variables (X)

X_train = train_df.drop(columns=['readmitted_binary'])
y_train = train_df['readmitted_binary']
X_test = test_df.drop(columns=['readmitted_binary'])
y_test = test_df['readmitted_binary']

In [ ]:
# Guardem info
import os
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Saved train/test splits to data/processed/")